# Integración y perfil de los datos
Este cuaderno documenta la lectura de los archivos SPSS del INE (divorcios y violencia intrafamiliar),
su perfilado inicial y la construcción del panel Departamento–Año que alimentará el resto del análisis.


### Objetivos
1. Detectar automáticamente el año de cada archivo `.sav` y perfilar ambos datasets.
2. Documentar formas, tipos de variables, faltantes y diccionario básico para cada fuente.
3. Limpiar nombres/códigos de departamento, garantizar unicidad y construir el panel final.
4. Persistir el resultado en `Datos/Procesados/panel_departamento_anio.csv` para reutilizarlo en todas las libretas.

In [ ]:
from pathlib import Path
import sys

import pandas as pd

pd.options.display.max_rows = 30
pd.options.display.max_columns = 20
pd.options.display.float_format = '{:,.2f}'.format

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name != 'MD_Proyecto1':
    for parent in PROJECT_ROOT.parents:
        if parent.name == 'MD_Proyecto1':
            PROJECT_ROOT = parent
            break
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

import data_pipeline as dp

In [2]:
div_raw = dp.load_divorcios_raw()
vif_raw, vif_labels = dp.load_vif_raw()

div_raw['anio'] = dp._coalesce_year(div_raw, ['anoocu', 'anoreg', 'anio_fuente'])
vif_raw['anio'] = dp._coalesce_year(vif_raw, ['hec_ano', 'ano_emision', 'anio_fuente'])

div_shape = div_raw.shape
vif_shape = vif_raw.shape
print(f'Divorcios: {div_shape[0]:,} filas, {div_shape[1]} columnas')
print(f'VIF: {vif_shape[0]:,} filas, {vif_shape[1]} columnas')
print('Años divorcios:', div_raw['anio'].dropna().astype(int).min(), '-', div_raw['anio'].dropna().astype(int).max())
print('Años VIF:', vif_raw['anio'].dropna().astype(int).min(), '-', vif_raw['anio'].dropna().astype(int).max())

Divorcios: 66,419 filas, 24 columnas
VIF: 365,129 filas, 79 columnas
Años divorcios: 2013 - 2022
Años VIF: 2000 - 2023


### Perfil del dataset de divorcios
- Se trabaja a nivel registro (acto) con columnas de registro y de ocurrencia.
- Años detectados automáticamente (columna `anio`).
- Se identifican variables numéricas vs. categóricas y los campos con mayor cantidad de nulos.

In [3]:
div_por_anio = (
    div_raw.dropna(subset=['anio'])
    .groupby('anio')
    .size()
    .reset_index(name='filas')
    .sort_values('anio')
)
display(div_por_anio.head(10))
display(div_por_anio.tail(10))

div_types = dp.split_types(div_raw)
print('Variables numéricas:', len(div_types['numericas']))
print('Variables categóricas:', len(div_types['categoricas']))

div_missing = dp.missing_summary(div_raw).head(10)
display(div_missing)

div_dictionary = dp.describe_columns(div_raw)
display(div_dictionary)

,anio,filas
0,2013,4377
1,2014,5392
2,2015,7074
3,2016,5665
4,2017,5808
5,2018,6255
6,2019,8203
7,2020,4074
8,2021,9621
9,2022,9950


,anio,filas
0,2013,4377
1,2014,5392
2,2015,7074
3,2016,5665
4,2017,5808
5,2018,6255
6,2019,8203
7,2020,4074
8,2021,9621
9,2022,9950


Variables numéricas: 4
Variables categóricas: 20


,columna,nulos,%_nulos
0,puehom,39626,59.66
1,puemuj,39626,59.66
2,ppermuj,26793,40.34
3,pperhom,26793,40.34
4,anio_fuente,16843,25.36
5,anoocu,11117,16.74


,columna,dtype,nulos,%_nulos,ejemplo
0,depreg,str,0,0.00,Guatemala
1,mupreg,str,0,0.00,Guatemala
2,mesreg,category,0,0.00,Diciembre
3,anoreg,float64,0,0.00,"2,016.00"
4,diaocu,float64,0,0.00,3.00
5,mesocu,category,0,0.00,Noviembre
6,anoocu,float64,11117,16.74,"2,016.00"
7,depocu,str,0,0.00,Guatemala
8,mupocu,str,0,0.00,Guatemala
9,edadhom,object,0,0.00,Ignorado


### Perfil del dataset de violencia intrafamiliar (VIF)
- Incluye variables de evento, víctima y agresor (53 columnas).
- Los años se detectan con `hec_ano` y las rutas se usan como respaldo.
- Se resumen los campos con mayor porcentaje de datos faltantes.

In [4]:
vif_por_anio = (
    vif_raw.dropna(subset=['anio'])
    .groupby('anio')
    .size()
    .reset_index(name='filas')
    .sort_values('anio')
)
display(vif_por_anio.head(10))
display(vif_por_anio.tail(10))

vif_types = dp.split_types(vif_raw)
print('Variables numéricas:', len(vif_types['numericas']))
print('Variables categóricas:', len(vif_types['categoricas']))

vif_missing = dp.missing_summary(vif_raw).head(12)
display(vif_missing)

vif_dictionary = dp.describe_columns(vif_raw)
display(vif_dictionary.head(30))

,anio,filas
0,2000,4
1,2001,18
2,2002,7
3,2003,8
4,2004,14
5,2005,11
6,2006,19
7,2007,8
8,2008,24
9,2009,35


,anio,filas
14,2014,33653
15,2015,31343
16,2016,30845
17,2017,29957
18,2018,29895
19,2019,31538
20,2020,28270
21,2021,36318
22,2022,36565
23,2023,35832


Variables numéricas: 76
Variables categóricas: 3


,columna,nulos,%_nulos
0,articulotras4,363038,99.43
1,articulotras3,363038,99.43
2,articulotras1,363038,99.43
3,articulotras2,363038,99.43
4,articulocodpen2,362180,99.19
5,articulocodpen1,362180,99.19
6,articulocodpen4,362180,99.19
7,articulocodpen3,362180,99.19
8,tipo_discaq,355246,97.29
9,filter,335141,91.79


,columna,dtype,nulos,%_nulos,ejemplo
0,hec_dia,float64,0,0.00,4.00
1,hec_mes,float64,0,0.00,2.00
2,hec_ano,float64,0,0.00,"2,014.00"
3,hec_deptomcpio,float64,0,0.00,"1,708.00"
4,hec_tipagre,float64,0,0.00,"1,122.00"
5,dia_emision,float64,0,0.00,5.00
6,mes_emision,float64,0,0.00,2.00
7,ano_emision,float64,0,0.00,"2,014.00"
8,depto_mcpio,float64,0,0.00,"1,101.00"
9,quien_reporta,float64,0,0.00,3.00


### Integración Departamento–Año
1. Normalización de nombres/códigos de departamento.
2. Conteos por sexo, tipo de agresión y grupo de edad (cuando existen).
3. Unión exterior para mantener combinaciones faltantes en cualquiera de las fuentes.

In [5]:
panel = dp.build_panel(force_rebuild=False)
panel_path = dp.PANEL_PATH
print(f'Filas panel: {len(panel):,} | Columnas: {panel.shape[1]}')
print('¿Departamentos únicos?', panel['departamento'].nunique())
print('¿Duplicados departamento-año?', panel[['departamento', 'anio']].duplicated().any())
display(panel.head())
display(panel[['divorcios_total', 'vif_total']].describe())
print(f'Archivo persistido en: {panel_path}')

Filas panel: 351 | Columnas: 120
¿Departamentos únicos? 22
¿Duplicados departamento-año? False


,departamento,anio,divorcios_total,vif_total,vif_sexo_hombres,vif_sexo_mujeres,vif_tipo_fisica,vif_tipo_fisica_patrimonial,vif_tipo_fisica_psicologica,vif_tipo_fisica_psicologica_patrimonial,...,vif_edad_90,vif_edad_91,vif_edad_92,vif_edad_93,vif_edad_94,vif_edad_95,vif_edad_96,vif_edad_97,vif_edad_98,vif_edad_99
0,Baja Verapaz,2000,NaN,1,0,1,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
1,Guatemala,2000,NaN,3,0,3,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,Chimaltenango,2001,NaN,1,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,Guatemala,2001,NaN,8,1,7,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
4,Izabal,2001,NaN,1,0,1,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0


,divorcios_total,vif_total
count,220.00,351.00
mean,301.90,"1,025.05"
std,503.80,"1,281.11"
min,39.00,1.00
25%,130.75,13.50
50%,174.00,718.00
75%,251.50,"1,490.00"
max,"3,491.00","8,381.00"


Archivo persistido en: C:\Users\viank\OneDrive\Desktop\Mineria_Datos\MD_Proyecto1\Datos\Procesados\panel_departamento_anio.csv
